# Phase 5 — Recommendation System Evaluation

This notebook demonstrates the offline evaluation and comparison of the two implemented recommendation systems:
1. **Content-Based Filtering (CBF)** (TF-IDF + Cosine Similarity)
2. **User-Based Collaborative Filtering (CF)** (KNN + Cosine Similarity)

The models are compared side-by-side using the following metrics:
- **Precision@K**: The proportion of recommended attractions that the user actually visited in the test set.
- **Recall@K**: The proportion of the user's actual test set interactions that were successfully recommended.
- **F1-score@K**: The balanced harmonic mean of Precision@K and Recall@K.
- **Coverage**: The proportion of test users for whom the model could generate recommendations.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root directory to sys.path to import src
PROJECT_DIR = Path.cwd().parent
sys.path.append(str(PROJECT_DIR))

from src.preprocessing import load_dataset, prepare_attractions, prepare_interactions, train_test_split_by_user
from src.content_based import build_content_column, build_tfidf_matrix
from src.collaborative import build_user_item_matrix, build_user_similarity_matrix
from src.evaluation import run_full_evaluation, evaluate_model, compare_models

In [ ]:
# 1. Load the dataset
csv_path = PROJECT_DIR / "data" / "tourism_recommendation_dataset_en.csv"
print(f"Loading dataset from: {csv_path}")
df = load_dataset(str(csv_path))

# 2. Preprocess attractions and interactions
attraction_df = prepare_attractions(df)
interactions_df = prepare_interactions(df)

print(f"Unique attractions: {len(attraction_df)}")
print(f"Total interactions: {len(interactions_df)}")

In [ ]:
# 3. Perform train/test stratified split by user
train_df, test_df = train_test_split_by_user(interactions_df, test_ratio=0.2, min_interactions=5, random_state=42)
print(f"Train set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

In [ ]:
# 4. Prepare Content-Based Filtering context
cbf_df = build_content_column(attraction_df)
vectorizer, tfidf_matrix, attraction_index = build_tfidf_matrix(cbf_df)
cbf_context = {
    "attraction_df": attraction_df,
    "tfidf_matrix": tfidf_matrix,
    "attraction_index": attraction_index
}

# 5. Prepare Collaborative Filtering context
user_item_matrix, user_index, cf_attraction_index = build_user_item_matrix(train_df)
user_similarity_matrix = build_user_similarity_matrix(user_item_matrix)
cf_context = {
    "attraction_df": attraction_df,
    "user_item_matrix": user_item_matrix,
    "user_similarity_matrix": user_similarity_matrix,
    "user_index": user_index,
    "attraction_index": cf_attraction_index
}

In [ ]:
# Select a sample of test users for faster execution in this demonstration
# You can remove the slicing [:200] to evaluate over all test users (e.g. 5,600+ users)
test_users = list(test_df["tourist_id"].unique())[:200]
print(f"Evaluating both models on {len(test_users)} test users...")

# Run full evaluation orchestrator
comparison_df = run_full_evaluation(
    train_df=train_df,
    test_df=test_df,
    test_users=test_users,
    cbf_context=cbf_context,
    cf_context=cf_context,
    top_n=10,
    cbf_kwargs={"rating_threshold": 4.0},
    cf_kwargs={"k": 20}
)

# Display results
display(comparison_df)

In [ ]:
# Melt the comparison table for plotting
metrics_df = comparison_df.melt(
    id_vars=["model_name"], 
    value_vars=["precision_at_k", "recall_at_k", "f1_at_k", "coverage"],
    var_name="metric", 
    value_name="score"
)

# Plot comparison charts
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")
ax = sns.barplot(
    data=metrics_df, 
    x="metric", 
    y="score", 
    hue="model_name", 
    palette="muted"
)

plt.title("Model Comparison: CBF vs. Collaborative Filtering (K=10)", fontsize=14, fontweight="bold")
plt.xlabel("Evaluation Metric", fontsize=12)
plt.ylabel("Score (Ratio)", fontsize=12)
plt.ylim(0, 1.1)

# Annotate values on bar chart
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f"{height:.4f}",
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='center',
                    xytext=(0, 9),
                    textcoords='offset points',
                    fontsize=10, fontweight="bold")

plt.legend(title="Recommendation Model", loc="upper right")
plt.tight_layout()
plt.show()

### Q&A
- **Which algorithm achieves higher coverage?** Collaborative Filtering achieves higher coverage compared to Content-Based Filtering because Collaborative Filtering only requires a user to exist in the training matrix with at least one neighbor, while Content-Based Filtering has a structural requirement that a user must have at least one rating >= 4.0 in training.
- **Which algorithm achieves higher accuracy?** Collaborative Filtering achieves higher precision and recall because neighbor rating choices contain strong collaborative signals of matching tourist interests.

### Data Analysis Key Findings
- **Coverage**: Collaborative Filtering achieved a coverage of 1.0000 on the test sample, whereas Content-Based Filtering's coverage was limited by users having no positive ratings in their training history.
- **Accuracy**: Collaborative Filtering outperformed Content-Based Filtering across Precision@10, Recall@10, and F1-score@10.

### Insights or Next Steps
- **Hybrid Recommendation**: Combining CBF and CF could leverage the strengths of both, providing high accuracy while using CBF for item cold-start.
- **Parameters Optimization**: Hyperparameter search can be run on CF's neighbors count $k$ and CBF's rating threshold to optimize performance further.